In [2]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study
import diff_cont
import lambda_cont
import libs.agent_infra as ai
import os

import json
import datetime
import itertools
import constants as Cs
import concurrent.futures
import numpy as np
from concurrent.futures import ProcessPoolExecutor

SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 6

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-30 19:46:49.080934: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-30 19:46:49.175966: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-30 19:46:51.484852: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF

In [ ]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 0.7, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="add_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
study = create_study(direction="maximize", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/add_novelty/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=170, n_jobs=5)
print(study.best_trials)

[I 2026-05-30 19:46:56,137] A new study created in RDB with name: no-name-be452702-b789-4c38-a988-6c169c699758
[W 2026-05-30 19:47:13,458] Trial 4 failed with parameters: {'lambda': 40, 'mutation_rate': 0.45, 'cross_rate': 0.5, 'start_fit_w': 0.30000000000000004} because of the following error: KeyboardInterrupt().
concurrent.futures.process._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/process.py", line 246, in _process_worker
    r = call_item.fn(*call_item.args, **call_item.kwargs)
  File "/home/schkliba/git/MastersThesis/diff_cont.py", line 133, in argumented_function
    alg.run()
  File "/home/schkliba/git/MastersThesis/containers.py", line 166, in run
    self.final_pop, self.logbook = de.differential_evolution(
  File "/home/schkliba/git/MastersThesis/differential_evo.py", line 43, in differential_evolution
    fitnesses = toolbox.map(toolbox.evaluate, population)
  File "/home/schkliba

KeyboardInterrupt: 